# Notes
- action names and order are consistent between the amass data file and the v3d file 
- videos sampled at 30 Hz, mocap at 120 Hz
- comparing T-1 to start-end yields perfect 4.0 ratio, I'm probably missing something obvious with the integer logic as why T-1... 
- mp4 videos seem to have an offset, but .avi potentially is trimmed, looks about right (still only compared 1 action)
- avi videos are from different cameras and angles -> pot. good for us to include for more data, and perhaps an offset can be calculated from the difference in time against the mp4 videos. 
- No nevermind, looks like they trimmed in both ends... 
- for data loader, clever use of memory to store different angles and somehow cleverly access same gt but different angles depending on index being called. 
- looks like CP2 mp4 files are also trimmed... 

In [1]:
import random
from pathlib import Path

import h5py
import numpy as np
import scipy.io as sio

p1 = "F_AMASS/F_amass_Subject_1.mat"
p10 = "F_AMASS/F_amass_Subject_10.mat"

m1 = sio.loadmat(str(p1), struct_as_record=False, squeeze_me=False)



In [2]:
def _unwrap(x):
    """
    Unwrap nested numpy object arrays from scipy.io.loadmat (squeeze_me=False).
    Only peels single-element (size==1) object arrays, so multi-element arrays
    like move (21,1) are left intact. Stops when we reach a mat_struct,
    a numeric ndarray, or a scalar -- regardless of how many layers deep.
    """
    while isinstance(x, np.ndarray) and x.dtype == object and x.size == 1:
        x = x.flat[0]
    return x


def _scalar(x) -> int | float:
    """Extract a Python scalar from any numpy array shape."""
    if isinstance(x, np.ndarray):
        return x.flat[0].item()
    return x


def _str(x) -> str:
    """Extract a plain Python str from a numpy string scalar or 1-element array."""
    if isinstance(x, np.ndarray):
        return str(x.flat[0])
    return str(x)


def load_amass_mat(path: Path) -> tuple[dict, list[dict]] | tuple[None, None]:
    """
    Load one F_amass_Subject_X.mat.

    Uses squeeze_me=False and explicit unwrapping to be robust across
    all scipy versions and all 90 subject files.

    Returns
    -------
    meta  : dict        — subject metadata
    clips : list[dict]  — one dict per action clip
    """
    try:
        mat = sio.loadmat(str(path), struct_as_record=False, squeeze_me=False)
    except Exception as e:
        print(f"Cannot load {path.name}: {e}")
        return None, None

    top_key = next(k for k in mat if not k.startswith("__"))
    subj    = _unwrap(mat[top_key])   # mat_struct with fields: id, subject, move
    s       = _unwrap(subj.subject)   # mat_struct with fields: id, sex, height, ...

    meta = {
        "id":     _str(s.id),
        "gender": _str(s.sex),
        "height": int(_scalar(s.height)),
        "mass":   int(_scalar(s.mass)),
        "age":    int(_scalar(s.age)),
    }

    # move_arr shape varies across subjects: (21,1), (1,21), etc.
    # Iterate .flat so we never assume a particular axis layout.
    move_arr = subj.move
    clips = []
    for cell in move_arr.flat:
        action = _unwrap(cell)

        # Some subjects have extra nesting layers — keep peeling
        # until we reach a mat_struct or give up
        max_depth = 10
        depth = 0
        while not hasattr(action, "_fieldnames") and depth < max_depth:
            if isinstance(action, np.ndarray) and action.size > 0:
                action = action.flat[0]
            else:
                break
            depth += 1

        if not hasattr(action, "_fieldnames"):
            print(f"  Could not unwrap action cell in {meta['id']}, skipping. "
                        f"Final type: {type(action).__name__}")
            continue

        poses = action.jointsExpMaps_amass.astype(np.float32)    # (T, 52, 3)
        trans = action.RootTranslation_amass.astype(np.float32)  # (T, 3)
        betas = action.jointsBetas_amass.astype(np.float32).reshape(16)  # (16,)

        clips.append({
            "action": _str(action.description),
            "poses":  poses,
            "trans":  trans,
            "betas":  betas,
            "T":      poses.shape[0],
        })

    return meta, clips

In [3]:
meta, clips = load_amass_mat("F_AMASS/F_amass_Subject_1.mat")
clips[0].keys()

dict_keys(['action', 'poses', 'trans', 'betas', 'T'])

In [4]:
clips[0].keys()
action_names = [clip['action'] for clip in clips]
action_names

['kicking',
 'dancing_rm',
 'pointing',
 'hand_clapping',
 'jumping_jack',
 'stretching',
 'crossarms',
 'running_in_spot',
 'crawling',
 'walking',
 'hand_waving',
 'checking_watch',
 'sideways',
 'vertical_jumping',
 'sitting_down',
 'taking_photo',
 'cross_legged_sitting',
 'throw/catch',
 'jogging',
 'scratching_head',
 'phone_talking']

In [5]:
path = "F_Subjects_1_45/F_v3d_Subject_1.mat"

mat = sio.loadmat(str(path), struct_as_record=False, squeeze_me=False)
top_key = next(k for k in mat if not k.startswith("__"))
subj    = _unwrap(mat[top_key])   # mat_struct with fields: id, subject, move
s       = _unwrap(subj.subject)   # mat_struct with fields: id, sex, height, ...

meta = {
    "id":     _str(s.id),
    "gender": _str(s.sex),
    "height": int(_scalar(s.height)),
    "mass":   int(_scalar(s.mass)),
    "age":    int(_scalar(s.age)),
}
mat
meta
subj.__dict__
subj.move
# mat[top_key].move
# subj.__dict__
# s.__dict__
# meta, clips = load_amass_mat(path)
move_arr = _unwrap(subj.move)
move_arr.__dict__
move_arr.flags30[0,1]
action_names_compare = [str(_unwrap(action[0])).lstrip("['").rstrip("']") for action in move_arr.motions_list]
action_names_compare

['kicking',
 'dancing_rm',
 'pointing',
 'hand_clapping',
 'jumping_jack',
 'stretching',
 'crossarms',
 'running_in_spot',
 'crawling',
 'walking',
 'hand_waving',
 'checking_watch',
 'sideways',
 'vertical_jumping',
 'sitting_down',
 'taking_photo',
 'cross_legged_sitting',
 'throw/catch',
 'jogging',
 'scratching_head',
 'phone_talking']

In [6]:
action_names == action_names_compare

True

In [7]:
int(move_arr.flags30[0,0])
action_inds = np.array([[int(tup[0]),int(tup[1])] for tup in move_arr.flags30])
len(action_inds),len(clips) # checking same length

(21, 21)

In [15]:
# Close to 4, but becomes 4.0 if we compare to T-1, maybe some difference in "up to and including" logic. 
for i in range(len(clips)):
    T = clips[i]['T']-1
    diff = action_inds[i][1]-action_inds[i][0]
    print(T)
    print(diff)
    print(T/diff)
    print()

580
145
4.0

444
111
4.0

288
72
4.0

340
85
4.0

416
104
4.0

436
109
4.0

364
91
4.0

424
106
4.0

812
203
4.0

704
176
4.0

308
77
4.0

536
134
4.0

684
171
4.0

456
114
4.0

984
246
4.0

444
111
4.0

1108
277
4.0

624
156
4.0

820
205
4.0

472
118
4.0

620
155
4.0



In [22]:
from decord import VideoReader, cpu
import matplotlib.pyplot as plt

video_path = "videos/PG1_avi/F_PG1_Subject_1_L.avi"

action_idx = 0
action = clips[action_idx]
print(f'Action: {action["action"]}')
video_flag = action_inds[action_idx]
sframe = video_flag[0]
eframe = video_flag[1]

gt_fps = 120
video_fps = 30

def frame_to_time(frame, fps):
    return frame/fps

def time_to_frame(time,fps):
    return int(round(time*fps,0))

def show_frame(vr, frame_idx):
    frame = vr[frame_idx].asnumpy()
    plt.figure(figsize=(6,6))
    plt.imshow(frame)
    plt.title(f"Frame {frame_idx}")
    plt.axis("off")
    plt.show()


vr = VideoReader(video_path, ctx=cpu(0))



# Metadata
num_frames = len(vr)
fps = vr.get_avg_fps()
duration = num_frames / fps
mins = duration // 60
secs = duration % 60
print(f"Total frames: {num_frames}")
print(f"FPS: {fps:.2f}")
print(f"Duration (s): {duration:.2f}")
print(f'Duration (m:s): {int(mins)}:{secs:.2f}')
vid_start = frame_to_time(sframe,video_fps)
new_frame = time_to_frame(27,video_fps) # eyeballing shows that action starts around 27 seconds, but vid_start is around 8 seconds, so 19 second offset... 

# show_frame(vr, sframe)
# show_frame(vr, new_frame)

# frame_to_time(1069, 30), vid_start, new_frame

vr[sframe:eframe].asnumpy().shape

Action: kicking
Total frames: 3441
FPS: 30.00
Duration (s): 114.70
Duration (m:s): 1:54.70


(145, 600, 800, 3)

In [ ]:
vr.__dict__


{'_handle': c_void_p(2930894393520),
 '_num_frame': 4369,
 '_key_indices': None,
 '_frame_pts': None,
 '_avg_fps': 30.00894979665485}

In [10]:
print("Frame 0 preview:")
show_frame(vr, 0)

print("Frame 200 preview:")
show_frame(vr, 200)

print("Frame 800 preview:")
show_frame(vr, 800)

Frame 0 preview:


NameError: name 'vr' is not defined

In [11]:
avi_path = "F_PG1_Subject_1_L.avi"
vr = VideoReader(avi_path, ctx=cpu(0))



# Metadata
num_frames = len(vr)
fps = vr.get_avg_fps()
duration = num_frames / fps
mins = duration // 60
secs = duration % 60
print(f"Total frames: {num_frames}")
print(f"FPS: {fps:.2f}")
print(f"Duration (s): {duration:.2f}")
print(f'Duration (m:s): {int(mins)}:{secs:.2f}')
vid_start = frame_to_time(sframe,video_fps)

show_frame(vr, sframe)
show_frame(vr, sframe+40)
show_frame(vr,eframe-60)


RuntimeError: Error reading F_PG1_Subject_1_L.avi...